In [0]:
-- ============================================================
-- FILE    : 03_gold_schema.sql
-- LAYER   : Gold
-- SCHEMA  : customer360_gold
-- PURPOSE : CREATE TABLE statements for all 5 Gold tables
--           Star schema — 4 dimensions + 1 fact
--           Optimised for Power BI — draw 4 relationships in
--           Model view after connecting via JDBC
-- ============================================================

CREATE SCHEMA IF NOT EXISTS customer360_gold
COMMENT 'Customer 360 — Gold layer (star schema for Power BI)';

-- ────────────────────────────────────────────────────────────
-- DIMENSION TABLES
-- ────────────────────────────────────────────────────────────

-- TABLE : customer360_gold.gold_dim_customer
-- GRAIN : One row per customer
-- NOTE  : Enriched from Silver dim_customer + dim_city.
--         Adds state, region, city_tier for Power BI slicers.
-- POWER BI: Use as slicer dimension — filter by city, age, gender
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_gold.gold_dim_customer (
    customer_key       STRING  NOT NULL  COMMENT 'PK — CUST_ + user_id',
    user_id            STRING            COMMENT 'NK — natural key',
    name               STRING            COMMENT 'Full name',
    age                INT               COMMENT 'Age at signup',
    gender             STRING            COMMENT 'Male / Female / Other',
    city               STRING            COMMENT 'City at signup',
    state              STRING            COMMENT 'Indian state — enriched from dim_city',
    region             STRING            COMMENT 'North / South / East / West',
    city_tier          STRING            COMMENT 'Tier 1 / Tier 2 — enriched from dim_city',
    country            STRING            COMMENT 'Always India',
    email              STRING            COMMENT 'Email address',
    signup_date        DATE              COMMENT 'Account creation date',
    customer_age_days  INT               COMMENT 'Days since signup',
    is_active          INT               COMMENT '1 = active  0 = inactive'
)
USING DELTA
COMMENT 'Customer dimension — identity and geography for Power BI demographic slicers';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_gold.gold_dim_date
-- GRAIN : One row per calendar date (reused from Silver)
-- NOTE  : Power BI needs a proper date table for time intelligence.
--         Connect first_txn_date in fact → date_key here.
-- POWER BI: Mark as Date Table in Model view for time intelligence
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_gold.gold_dim_date (
    date_key              STRING  NOT NULL  COMMENT 'PK — YYYY-MM-DD',
    full_date             DATE              COMMENT 'Full date as DATE type',
    day_of_month          INT               COMMENT '1-31',
    day_of_week           INT               COMMENT '1=Monday 7=Sunday',
    day_name              STRING            COMMENT 'Monday / Tuesday ... Sunday',
    week_number           INT               COMMENT 'ISO week number',
    month                 INT               COMMENT '1-12',
    month_name            STRING            COMMENT 'January ... December',
    quarter               INT               COMMENT '1-4',
    quarter_label         STRING            COMMENT 'Q1 / Q2 / Q3 / Q4',
    year                  INT               COMMENT 'Calendar year',
    fiscal_quarter_label  STRING            COMMENT '2024-Q1 format',
    is_weekend            INT               COMMENT '1 = weekend  0 = weekday',
    is_holiday            INT               COMMENT '1 = Indian public holiday',
    is_non_working_day    INT               COMMENT '1 = weekend or holiday'
)
USING DELTA
COMMENT 'Date dimension — full spine 2023-2026 for Power BI time intelligence';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_gold.gold_dim_segment
-- GRAIN : One row per segment (4 rows total)
-- NOTE  : Enriched with description and target_channel.
--         Tells the marketing team HOW to reach each segment.
-- POWER BI: Use as slicer — filter entire report by segment
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_gold.gold_dim_segment (
    segment_key     STRING  NOT NULL  COMMENT 'PK — SEG_PREMIUM / SEG_STANDARD / SEG_BASIC / SEG_TRIAL',
    segment_name    STRING            COMMENT 'Premium / Standard / Basic / Trial',
    description     STRING            COMMENT 'Human-readable segment description',
    target_channel  STRING            COMMENT 'Account Manager / Email / Push Notification / In-App Message'
)
USING DELTA
COMMENT 'Segment master — 4 segments with marketing channel guidance';

-- ────────────────────────────────────────────────────────────
-- TABLE : customer360_gold.gold_dim_tier
-- GRAIN : One row per customer tier (5 rows total)
-- NOTE  : Derived from RFM score. retention_strategy tells the
--         business WHAT to do with each tier — actionable Gold.
-- POWER BI: Use as slicer — filter by tier to see Champions only etc
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_gold.gold_dim_tier (
    tier_key             STRING  NOT NULL  COMMENT 'PK — TIER_CHAMPIONS / TIER_LOYAL / TIER_POTENTIAL / TIER_AT_RISK / TIER_DORMANT',
    tier_name            STRING            COMMENT 'Champions / Loyal / Potential / At Risk / Dormant',
    rfm_range            STRING            COMMENT 'RFM score range e.g. 13-15 for Champions',
    retention_strategy   STRING            COMMENT 'Recommended action for CRM / retention team',
    priority_rank        INT               COMMENT '1=Champions (highest priority) to 5=Dormant'
)
USING DELTA
COMMENT 'Customer tier master — 5 tiers with RFM range and retention strategy';

-- ────────────────────────────────────────────────────────────
-- FACT TABLE
-- ────────────────────────────────────────────────────────────

-- TABLE : customer360_gold.gold_fact_customer_metrics
-- GRAIN : One row per customer — all metrics aggregated from Silver
-- NOTE  : Rebuilt on every pipeline run (overwrite).
--         All nulls filled with 0 — no nulls in this table.
--
-- POWER BI RELATIONSHIPS (draw these in Model view):
--   gold_fact_customer_metrics.customer_key  → gold_dim_customer.customer_key  (many-to-one)
--   gold_fact_customer_metrics.segment_key   → gold_dim_segment.segment_key    (many-to-one)
--   gold_fact_customer_metrics.tier_key      → gold_dim_tier.tier_key          (many-to-one)
--   gold_fact_customer_metrics.first_txn_date → gold_dim_date.date_key         (many-to-one)
-- ────────────────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS customer360_gold.gold_fact_customer_metrics (

    -- ── Foreign keys (Power BI relationships) ────────────
    customer_key     STRING  NOT NULL  COMMENT 'FK → gold_dim_customer.customer_key',
    segment_key      STRING            COMMENT 'FK → gold_dim_segment.segment_key',
    tier_key         STRING            COMMENT 'FK → gold_dim_tier.tier_key',
    first_txn_date   STRING            COMMENT 'FK → gold_dim_date.date_key — date of first transaction',

    -- ── Transaction metrics ───────────────────────────────
    total_spend      DOUBLE            COMMENT 'Lifetime value — sum of net_amount on successful transactions only',
    txn_count        LONG              COMMENT 'Total transaction count including failed and refunded',
    avg_order_value  DOUBLE            COMMENT 'Average net_amount on successful transactions',
    successful_txns  LONG              COMMENT 'Count of status=success transactions',
    failed_txns      LONG              COMMENT 'Count of status=failed transactions',
    refunded_txns    LONG              COMMENT 'Count of status=refunded transactions',
    failure_rate     DOUBLE            COMMENT 'failed_txns / txn_count * 100 — percentage',
    last_txn_date    STRING            COMMENT 'Date of most recent transaction YYYY-MM-DD',
    top_category     STRING            COMMENT 'Most frequent purchase category',

    -- ── Recency ───────────────────────────────────────────
    recency_days     INT               COMMENT 'Days since last transaction — recomputed every run against current_date(). 9999 if never transacted',

    -- ── RFM scores ────────────────────────────────────────
    r_score          INT               COMMENT 'Recency score 1-5 — 5=transacted in last 7 days  1=90+ days',
    f_score          INT               COMMENT 'Frequency score 1-5 — 5=20+ transactions  1=1 transaction',
    m_score          INT               COMMENT 'Monetary score 1-5 — 5=INR 50k+  1=under INR 1k',
    rfm_score        INT               COMMENT 'Combined RFM score 3-15 (r+f+m)',

    -- ── Engagement metrics ────────────────────────────────
    session_count    LONG              COMMENT 'Total app sessions',
    avg_session_mins DOUBLE            COMMENT 'Average session duration in minutes',
    total_pages      LONG              COMMENT 'Total pages visited across all sessions',
    total_actions    LONG              COMMENT 'Total actions taken across all sessions',
    bounce_rate      DOUBLE            COMMENT 'Percentage of sessions under 60 seconds',
    engagement_score DOUBLE            COMMENT 'Composite score 0-100 — weighted: sessions 40% / actions 30% / pages 20% / non-bounce 10%',
    last_session_date STRING           COMMENT 'Date of most recent session',

    -- ── Support metrics ───────────────────────────────────
    ticket_count          LONG         COMMENT 'Total support tickets raised',
    open_tickets          LONG         COMMENT 'Currently unresolved tickets',
    critical_tickets      LONG         COMMENT 'Tickets with priority=Critical',
    avg_resolution_hrs    DOUBLE       COMMENT 'Average hours from ticket creation to resolution',
    avg_satisfaction      DOUBLE       COMMENT 'Average satisfaction score 1-5 on resolved tickets',
    support_health_score  DOUBLE       COMMENT 'Score 0-100 — penalises high/open/critical tickets, rewards fast resolution and high satisfaction',
    last_ticket_date      STRING       COMMENT 'Date of most recent support ticket',

    -- ── Churn ─────────────────────────────────────────────
    churn_flag    INT                  COMMENT '1 = churned  0 = active. Conditions: never transacted OR inactive 60+ days OR failure_rate > 40% OR low RFM + high tickets',
    churn_reason  STRING               COMMENT 'Never transacted / Inactive 60+ days / High payment failure rate / Low RFM + high support load — NULL if not churned',

    -- ── Lineage ───────────────────────────────────────────
    pipeline_run_id  STRING            COMMENT 'RUN_ID of pipeline that last rebuilt this table'
)
USING DELTA
COMMENT 'Customer metrics fact — one row per customer with all KPIs. Rebuilt on every pipeline run.';